# Verify revised 22.91 dB checkpoint with LPIPS
Runs one consistent 5-step DDIM evaluation with ARR disabled and seed 42.

In [ ]:
!pip install -q lpips scikit-image


In [ ]:
from pathlib import Path
import glob, hashlib, json, shutil, subprocess, sys, zipfile
import pandas as pd

EXPECTED_SHA256 = '5d7bac873b0b915fe6c0679b103fea9afe25f70de3958cab8da3d8779d156a37'
EXPECTED_PSNR = 22.909448355896885
EXPECTED_SSIM = 0.8145820149388322

source_zips = list(Path('/kaggle/input').rglob('lumidiff_2291_eval_source.zip'))
if len(source_zips) != 1:
    raise RuntimeError(f'Expected exactly one source ZIP, found: {source_zips}')
CODE = Path('/kaggle/working/lumidiff_2291_eval_source')
if CODE.exists():
    shutil.rmtree(CODE)
with zipfile.ZipFile(source_zips[0]) as zf:
    zf.extractall(CODE)

ckpts = list(Path('/kaggle/input').rglob('student_distilled.pth'))
matches = []
for path in ckpts:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(path, digest)
    if digest == EXPECTED_SHA256:
        matches.append(path)
if len(matches) != 1:
    raise RuntimeError(f'Expected one exact revised checkpoint, found: {matches}')
CHECKPOINT = matches[0]

low_dirs = [p for p in Path('/kaggle/input').rglob('Low') if str(p).endswith('Real_captured/Test/Low')]
pairs = []
for low in low_dirs:
    high = low.parent / 'Normal'
    if high.is_dir():
        pairs.append((low, high))
if len(pairs) != 1:
    raise RuntimeError(f'Expected one LOL-v2 Real test pair, found: {pairs}')
LOW, HIGH = pairs[0]
image_exts = {'.png', '.jpg', '.jpeg', '.bmp'}
n_low = sum(p.suffix.lower() in image_exts for p in LOW.iterdir())
n_high = sum(p.suffix.lower() in image_exts for p in HIGH.iterdir())
if (n_low, n_high) != (100, 100):
    raise RuntimeError(f'Expected 100/100 test images, found {n_low}/{n_high}')
print('Code:', CODE)
print('Checkpoint:', CHECKPOINT)
print('Test folders:', LOW, HIGH)


In [ ]:
OUT = Path('/kaggle/working/lumidiff_2291_lpips')
if OUT.exists():
    shutil.rmtree(OUT)
cmd = [
    sys.executable, 'evaluation.py',
    '--splits', f'lolv2_real:{LOW}:{HIGH}',
    '--inference-steps', '5',
    '--sampler', 'ddim',
    '--checkpoint', str(CHECKPOINT),
    '--gate-alpha', '0.0',
    '--gate-floor', '0.5',
    '--results-root', str(OUT),
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=CODE, check=True)


In [ ]:
summary_path = OUT / 'summary.csv'
per_image_path = OUT / 'per_image.csv'
summary = pd.read_csv(summary_path)
display(summary)
row = summary.iloc[0]
if abs(float(row.psnr_mean) - EXPECTED_PSNR) > 0.02:
    raise RuntimeError(f'PSNR did not reproduce: {row.psnr_mean}')
if abs(float(row.ssim_mean) - EXPECTED_SSIM) > 0.0002:
    raise RuntimeError(f'SSIM did not reproduce: {row.ssim_mean}')
if pd.isna(row.lpips_mean):
    raise RuntimeError('LPIPS is missing; do not use this run in the paper')
manifest = {
    'checkpoint': str(CHECKPOINT),
    'checkpoint_sha256': EXPECTED_SHA256,
    'dataset_low': str(LOW),
    'dataset_high': str(HIGH),
    'n': 100, 'seed': 42, 'sampler': 'ddim', 'steps': 5,
    'gate_alpha': 0.0, 'gate_floor': 0.5, 'lpips_backbone': 'alex',
    'command': cmd,
}
manifest_path = OUT / 'run_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))
evidence = Path('/kaggle/working/lumidiff_2291_lpips_evidence')
if evidence.exists():
    shutil.rmtree(evidence)
evidence.mkdir()
for path in (summary_path, per_image_path, manifest_path):
    shutil.copy2(path, evidence / path.name)
archive = shutil.make_archive(str(evidence), 'zip', evidence)
print('Download:', archive)
